# Multi-Modal RAG with Bedrock Managed Knowledge Bases & Smart Parsing

This notebook demonstrates **multi-modal RAG** using a Bedrock Managed Knowledge Base with **Smart Parsing** — ingest PDFs with charts, audio files, and video, then query across all modalities.

### What makes this multi-modal?

| Component | Role |
|-----------|------|
| **Smart Parsing** | Automatically extracts text from PDFs (including tables/charts), transcribes audio, and analyzes video |
| **Managed embedding (default)** | Embeds the extracted text representations into the vector store |
| **AgenticRetrieveStream** | Uses Claude to reason over chunks from all modalities (with `generate_response=True`) |

The key insight: **Smart Parsing converts all modalities to text** — you don't need a multimodal embedding model. Just point BMKB at your files and query.

### Multi-modal content in this notebook

| File | Type | What Smart Parsing extracts |
|------|------|----------------------------|
| `IF12695.pdf` | PDF with charts & tables | Text + table structure + chart/figure descriptions |
| `podcastdemo.mp3` | Audio | Transcribed speech → searchable text |
| `bda.m4v` | Video | Transcribed audio track + visual frame descriptions |

### Why BMKB vs DIY multi-modal RAG?

| DIY Pipeline | BMKB + Smart Parsing |
|---|---|
| Set up BDA for PDF parsing | Smart Parsing handles PDFs automatically |
| Set up Amazon Transcribe for audio | Smart Parsing transcribes audio automatically |
| Set up Rekognition for video frames | Smart Parsing analyzes video automatically |
| Provision a vector store (OpenSearch, etc.) | Bedrock manages the vector store |
| Build custom chunking per modality | Automatic chunking for all content types |
| **Result: Same quality, zero infrastructure** | |

### Architecture

```
┌─────────────┐   ┌───────────┐   ┌───────────┐
│ PDF (charts) │   │ Audio     │   │ Video     │
│ IF12695.pdf  │   │ .mp3      │   │ .m4v      │
└──────┬───────┘   └─────┬─────┘   └─────┬─────┘
       │                 │               │
       └─────────────────┼───────────────┘
                         ▼
              ┌─────────────────────────┐
              │     Smart Parsing       │ ← Converts all modalities to text
              │  (PDF → text + tables)  │
              │  (Audio → transcript)   │
              │  (Video → transcript)   │
              └───────────┬─────────────┘
                          ▼
              ┌─────────────────────────┐
              │   Managed embedding (default)   │ ← Standard text embeddings
              └───────────┬─────────────┘
                          ▼
              ┌─────────────────────────┐
              │  Managed Vector Store   │ ← No infrastructure needed
              └───────────┬─────────────┘
                          ▼
               Retrieve / AgenticRetrieveStream
```

### Prerequisites

- AWS credentials with Bedrock, IAM, and S3 permissions
- Model access enabled for **Managed embedding (default)** and **Claude**
- Python 3.10+

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Configuration

In [ ]:
import boto3
import sys
import time
import os
import json
import logging
import pprint
import urllib.request

sys.path.insert(0, "../..")

s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration ─────────────────────────────────────────────────────
knowledge_base_name = f'bmkb-multimodal-{suffix}'
bucket_name = f'bedrock-bmkb-multimodal-{suffix}-{account_id}'

# Embedding model — Smart Parsing converts all modalities to text,
# so standard text embeddings work for multi-modal RAG
embedding_model = 'amazon.titan-embed-text-v2:0'

# Generation model
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:     {region}')
print(f'Account:    {account_id}')
print(f'KB Name:    {knowledge_base_name}')
print(f'Bucket:     {bucket_name}')
print(f'Embedding:  {embedding_model}')
print(f'Generation: {generation_model_arn}')

## Step 2 — Prepare multi-modal data

Upload three types of content:
1. **PDF with charts/tables** — CRS report with visual elements
2. **Audio** — Podcast clip (MP3)
3. **Video** — Demo video (M4V)

In [ ]:
# Create S3 bucket
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Bucket {bucket_name} already exists')
except Exception:
    print(f'Creating bucket {bucket_name}')
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})

In [ ]:
# CRS report PDF — contains charts, tables, and visual content
# This file must be downloaded manually (the site blocks programmatic access).
#
# Download from your browser: https://sgp.fas.org/crs/misc/IF12695.pdf
# Save to: synthetic_dataset/IF12695.pdf
#
# If you want to use a different PDF with visual content, update the path below.

crs_pdf_path = '../../synthetic_dataset/IF12695.pdf'

if os.path.exists(crs_pdf_path) and os.path.getsize(crs_pdf_path) > 1000:
    size_kb = os.path.getsize(crs_pdf_path) / 1024
    print(f'✓ CRS report ready: {crs_pdf_path} ({size_kb:.1f} KB)')
else:
    print('⚠️  CRS report not found or empty.')
    print('    Download from: https://sgp.fas.org/crs/misc/IF12695.pdf')
    print(f'    Save to: {os.path.abspath(crs_pdf_path)}')
    print('    Then re-run this cell.')

In [ ]:
# Upload all multi-modal content to S3
multimodal_files = [
    '../../synthetic_dataset/IF12695.pdf',           # PDF with charts and tables
    '../../synthetic_dataset/podcastdemo.mp3',       # Audio file
    '../../synthetic_dataset/bda.m4v',               # Video file
]

print('Uploading multi-modal content to S3...')
for file_path in multimodal_files:
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f'  {file_name} ({size_mb:.1f} MB) → s3://{bucket_name}/{file_name}')
        s3_client.upload_file(file_path, bucket_name, file_name)
    else:
        print(f'  ⚠️  {file_path} not found — skipping')

# List uploaded files
print(f'\nFiles in bucket:')
resp = s3_client.list_objects_v2(Bucket=bucket_name)
for obj in resp.get('Contents', []):
    ext = obj['Key'].split('.')[-1].upper()
    size_mb = obj['Size'] / (1024 * 1024)
    modality = {'PDF': '📄', 'MP3': '🎵', 'M4V': '🎬', 'MP4': '🎬'}.get(ext, '📎')
    print(f'  {modality} {obj["Key"]} ({size_mb:.1f} MB)')

## Step 3 — Create BMKB with Smart Parsing

Smart Parsing is the default (and only) parsing strategy for BMKB. It handles all file types automatically:
- **PDFs** → extracts text, detects and describes charts/tables
- **Audio (MP3)** → transcribes speech to text
- **Video (M4V/MP4)** → transcribes audio track + describes visual content

All extracted content becomes text chunks that Managed embedding (default) can embed.

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    bucket_name=bucket_name,
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')
print(f'DS ID: {kb.ds_id}')

kb_id = kb.kb_id
%store kb_id

## Step 4 — Ingest multi-modal content

Smart Parsing processes each file type:
- PDF → text + table/chart descriptions
- MP3 → speech transcript
- M4V → audio transcript + visual descriptions

In [ ]:
time.sleep(30)
job = kb.start_ingestion_job()

## Step 5 — Query PDF content (charts and tables)

Ask questions about the visual content in the CRS PDF. Smart Parsing converted charts and tables into text descriptions that are now searchable.

In [ ]:
pdf_queries = [
    'What data is shown in the charts and figures?',
    'What are the key statistics from the tables?',
    'Describe the trends shown in the graphs.',
]

print('=== PDF Visual Content Queries ===')
for query in pdf_queries:
    print(f'\nQ: {query}')
    response = kb.retrieve(query, num_results=3)
    for i, res in enumerate(response.get('retrievalResults', []), 1):
        score = res['score']
        text = res['content']['text'][:200]
        print(f'  {i}. score={score:.4f} | {text}...')

## Step 6 — Query audio content

Ask about the podcast. Smart Parsing transcribed the MP3 — the transcript is now searchable.

In [ ]:
audio_queries = [
    'What topics are discussed in the podcast?',
    'What are the main points from the audio recording?',
    'Summarize what was said in the podcast.',
]

print('=== Audio Content Queries (from MP3) ===')
for query in audio_queries:
    print(f'\nQ: {query}')
    response = kb.retrieve(query, num_results=3)
    for i, res in enumerate(response.get('retrievalResults', []), 1):
        score = res['score']
        text = res['content']['text'][:200]
        print(f'  {i}. score={score:.4f} | {text}...')

## Step 7 — Query video content

Ask about the video demo. Smart Parsing transcribed the audio and described the visual frames.

In [ ]:
video_queries = [
    'What is demonstrated in the video?',
    'What is shown in the video demo?',
    'Describe what happens in the video.',
]

print('=== Video Content Queries (from M4V) ===')
for query in video_queries:
    print(f'\nQ: {query}')
    response = kb.retrieve(query, num_results=3)
    for i, res in enumerate(response.get('retrievalResults', []), 1):
        score = res['score']
        text = res['content']['text'][:200]
        print(f'  {i}. score={score:.4f} | {text}...')

## Step 8 — Cross-modal AgenticRetrieveStream (with generation)

Ask a question that requires synthesizing from all modalities. AgenticRetrieveStream with `generate_response=True` decomposes the query, retrieves chunks from PDF, audio, and video, and generates a unified answer.

In [ ]:
query = 'Based on all the documents, audio, and video content, what are the main topics covered and key takeaways?'

result = kb.agentic_retrieve_stream(
    query=query,
    model_arn=generation_model_arn,
    generate_response=True,
)

print('=== Cross-Modal AgenticRetrieveStream (with generation) ===')
print(f"Answer:\n{result['generated_response']['answer']}")
print(f"\nCitations: {len(result['generated_response'].get('citations', []))}")


In [ ]:
# Query specifically about visual/chart content from the PDF
query = 'What do the charts and graphs in the PDF document show? Describe the data and trends.'

result = kb.agentic_retrieve_stream(
    query=query,
    model_arn=generation_model_arn,
    generate_response=True,
)
print(result['generated_response']['answer'])


## Step 9 — Analyze chunk types by source

Retrieve a broad set of chunks to see what Smart Parsing extracted from each file.

In [ ]:
response = kb.retrieve('information data content topics', num_results=15)

print('=== Chunk Analysis by Source ===')
print(f'Total chunks: {len(response.get("retrievalResults", []))}\n')

for i, res in enumerate(response.get('retrievalResults', []), 1):
    text = res['content']['text']
    score = res['score']
    uri = res.get('location', {}).get('s3Location', {}).get('uri', '')
    source_file = uri.split('/')[-1] if uri else 'unknown'
    
    # Identify modality from source
    ext = source_file.split('.')[-1].lower() if '.' in source_file else ''
    modality = {'pdf': '📄 PDF', 'mp3': '🎵 AUDIO', 'm4v': '🎬 VIDEO', 'mp4': '🎬 VIDEO'}.get(ext, '📎 OTHER')
    
    print(f'{i}. [{modality}] score={score:.4f} | source={source_file}')
    print(f'   {text[:200]}...')
    print()

## Step 10 — AgenticRetrieveStream across modalities

In [ ]:
result = kb.agentic_retrieve_stream(
    query='Compare the information from the PDF, the podcast, and the video. What are the connections between them?',
    model_arn=generation_model_arn,
    max_results=15,
    max_iterations=3,
)

print('=== AgenticRetrieveStream (cross-modal) ===')
print(f'Trace events: {len(result["traces"])}')
for t in result['traces']:
    attrs = t.get('attributes', {})
    print(f'  [{attrs.get("step", "?")}] {attrs.get("status", "")}')

print(f'\nFinal: {len(result["results"])} deduplicated chunks')
for i, r in enumerate(result['results'][:7], 1):
    text = r['content']['text'][:150]
    print(f'  {i}. {text}...')
# Show generated response if available
if result.get('generated_response'):
    print(f"\nGenerated Answer:\n{result['generated_response']['answer']}")
    print(f"Citations: {len(result['generated_response'].get('citations', []))}")


## Step 11 — Cleanup

> Only run when done experimenting.

In [ ]:
# Uncomment to delete everything
print('===============================Deleting Knowledge Base and associated resources==============================')
# kb.delete_kb(delete_iam=True, delete_s3_bucket=True)

## Summary

| What | Details |
|---|---|
| KB Type | `MANAGED` — Bedrock handles the vector store |
| Parsing | **Smart Parsing** — automatic multi-modal content extraction |
| Embedding | Managed embedding (default) (text-only — Smart Parsing converts everything to text) |
| Data | PDF (charts/tables), MP3 (podcast), M4V (video demo) |
| Retrieval | Retrieve (raw chunks), AgenticRetrieveStream (agentic retrieval with optional generation) |

### What Smart Parsing handles automatically

| Input | What gets extracted | Embedding approach |
|-------|-------------------|-------------------|
| PDF with charts | Text + chart/table descriptions | Text embedding of descriptions |
| Audio (MP3) | Speech transcript | Text embedding of transcript |
| Video (M4V/MP4) | Audio transcript + frame descriptions | Text embedding of all extracted text |
| Scanned PDF | OCR + layout detection | Text embedding of OCR output |

### Key takeaway

With BMKB + Smart Parsing, you get **multi-modal RAG across PDF, audio, and video** with:
- No BDA (Bedrock Data Automation) setup
- No Amazon Transcribe pipeline for audio
- No Rekognition pipeline for video/images
- No separate vector stores per modality
- No multimodal embedding model needed (Smart Parsing converts to text)

Just upload your files and query. Bedrock handles the rest.